# D4 — Pembekuan dan evaluasi final di UJI — Anggota D

⚠️ **Baca dulu.** Notebook ini membuka data UJI. Data UJI hanya boleh dibuka **satu kali per run**, setelah kelima keputusan kelompok (K1–K5) disepakati.

Alurnya dua tahap:
1. **Tahap 1, pembekuan.** Membuat `PEMBEKUAN_UJI.json`. File ini berisi SHA-256 dataset, SHA-256 13 berkas kode dan model, serta nama yang menyetujui. **Commit dan push file ini dulu** sebelum lanjut ke tahap 2.
2. **Tahap 2, UJI.** Menjalankan 7 run yang terdaftar di manifest (`scripts/anggota_d/d4_uji_final.py --final`). Setiap akses tercatat di `uji_log.csv`.

Jalankan **kedua tahap di sesi Colab yang sama** dan dari zip repo yang sama. Sidik berkas dihitung dari file di sesi ini. Kalau repo diunggah ulang dari Windows, akhir baris file `.py` bisa berubah, dan kunci akan (dengan benar) menolak.

Waktu jalan tahap 2 sekitar 5–8 menit di Colab gratis.

In [ ]:
%pip -q install scikit-learn==1.6.1 pandas numpy scipy matplotlib joblib

## 0. Ambil repo
Repo harus sudah berisi paket D4 dan draf `results/anggota_d/PEMBEKUAN_UJI_DRAF.json` dengan kelima keputusan `DISEPAKATI`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = ""   # contoh: "https://github.com/<akun>/Sistem-Deteksi-Berita-Hoaks-Berbahasa-Indonesia.git"
BRANCH = "main"

work = Path("/content/repo") if Path("/content").exists() else Path("repo")
TARGET = "d4_uji_final.py"

def extract(zpath, dest):
    """Ekstrak zip; pemisah '\\' dari zip buatan Windows diubah menjadi '/'."""
    with zipfile.ZipFile(zpath) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            if name.endswith("/") or "__MACOSX" in name:
                continue
            out = dest / name
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)

def find_repo():
    hits = [p for p in work.rglob(TARGET) if p.parent.name == "anggota_d" and p.parent.parent.name == "scripts"]
    return hits[0].parents[2] if hits else None

# Bersihkan sisa percobaan sebelumnya yang gagal
if work.exists() and find_repo() is None:
    shutil.rmtree(work)

if not work.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(work)], check=True)
    else:
        from google.colab import files
        up = files.upload()
        work.mkdir(parents=True)
        extract(next(iter(up)), work)

REPO_DIR = find_repo()
if REPO_DIR is None:
    print("Isi teratas zip/repo:", sorted(p.name for p in work.iterdir())[:20])
    for name in ["evaluate.py", "test_evaluate.py", TARGET, "D4_UJI_Final.ipynb", "B2_run_experiments.py"]:
        hits = list(work.rglob(name))
        print(f"  {name:32s}", "ADA: " + str(hits[0].relative_to(work)) if hits else "TIDAK ADA")
    raise FileNotFoundError(
        "scripts/anggota_d/d4_uji_final.py tidak ada. Kemungkinan besar isi PAKET_D4 "
        "belum diekstrak ke root repo sebelum repo di-zip / di-push. Ekstrak paket itu di root repo lalu ulangi.")
assert (REPO_DIR / "prototipe/anggota_c/src/evaluate.py").is_file(), "prototipe/anggota_c/src/evaluate.py belum ada di repo"
assert (REPO_DIR / "notebooks/anggota_b/scripts/B2_run_experiments.py").is_file(), "skrip B tidak ada di repo"
print("Repo:", REPO_DIR)

## 0b. Google Drive dan dataset V3.1 milik A

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Bukan Google Colab: gunakan lokasi CSV di perangkat ini.")

DATASET_PATH = "/content/drive/MyDrive/Kelompok 5 - PSP/DATASET_V3_1_KANDIDAT_OPERASIONAL.csv"
assert Path(DATASET_PATH).is_file(), f"CSV tidak ditemukan. Ubah DATASET_PATH: {DATASET_PATH}"
D = REPO_DIR / "results" / "anggota_d"
def run(*args):
    r = subprocess.run([sys.executable, *map(str, args)], cwd=REPO_DIR, capture_output=True, text=True)
    print(r.stdout[-8000:]); print(r.stderr[-3000:]) if r.returncode not in (0,) else None
    return r.returncode

# Tahap 1 — Pembekuan
### 1a. Periksa draf (tidak menulis apa pun)
Harus berakhir dengan `[OK] Draf memenuhi syarat`.

In [ ]:
run("scripts/anggota_d/d2_bekukan.py", "--dataset", DATASET_PATH, "--cek",
    "--disetujui", "isi_nanti")   # nama di sini hanya untuk pengecekan

### 1b. Bekukan
Isi `DISETUJUI` dengan nama semua anggota yang menyetujui. Sel ini hanya bisa berhasil sekali.

In [ ]:
DISETUJUI = "Anggota A (nama), Anggota B (nama), Anggota C (nama), Anggota D (nama)"   # <- ubah

if (D / "PEMBEKUAN_UJI.json").exists():
    print("Manifest sudah ada, tidak dibekukan ulang.")
else:
    assert "(nama)" not in DISETUJUI, "Isi nama yang menyetujui dulu"
    run("scripts/anggota_d/d2_bekukan.py", "--dataset", DATASET_PATH, "--disetujui", DISETUJUI)
run("scripts/anggota_d/d2_status_uji.py", "--dataset", DATASET_PATH)

### 1c. Unduh manifest, lalu commit dan push dari laptop
1. Unduh `PEMBEKUAN_UJI.json`.
2. Salin ke `results/anggota_d/` di repo laptop.
3. Commit dengan pesan `chore(anggota_d): bekukan konfigurasi sebelum UJI`, lalu push.

**Lakukan ini sebelum menjalankan tahap 2.** Commit ini menjadi bukti bahwa pembekuan terjadi sebelum UJI dibuka.

In [ ]:
try:
    from google.colab import files
    files.download(str(D / "PEMBEKUAN_UJI.json"))
except ImportError:
    print("Salin manual:", D / "PEMBEKUAN_UJI.json")

# Tahap 2 — Evaluasi final di UJI
### 2a. Rencana (UJI belum dibuka)

In [ ]:
run("scripts/anggota_d/d4_uji_final.py", "--dataset", DATASET_PATH, "--rencana")

### 2b. Jalankan UJI
Ubah `SUDAH_COMMIT_MANIFEST` menjadi `True` **setelah** manifest di-commit dan di-push, lalu jalankan sel ini **sekali saja**.

- Kalau Colab terputus di tengah jalan, jalankan sel ini lagi. Run yang sudah `SELESAI` otomatis dilewati, dan run yang `GAGAL` boleh diulang.
- Jangan memakai `--alasan-ulang` kecuali hasil run yang sudah selesai benar-benar hilang. Pengulangan seperti itu wajib dilaporkan.

In [ ]:
SUDAH_COMMIT_MANIFEST = False   # <- ubah ke True setelah commit & push manifest
OPERATOR = "Anggota D (nama)"   # <- nama yang menjalankan

assert SUDAH_COMMIT_MANIFEST, "Commit dan push PEMBEKUAN_UJI.json dulu (tahap 1c)"
assert "(nama)" not in OPERATOR, "Isi nama operator"
rc = run("scripts/anggota_d/d4_uji_final.py", "--dataset", DATASET_PATH, "--final", "--operator", OPERATOR)
print("Kode keluar:", rc)

### 2c. Ringkasan hasil UJI

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 40)
display(pd.read_csv(D / "D4_UJI_RINGKASAN.csv")[["run_key","split","text_col","model","n_uji","macro_f1","macro_f1_ci_95","f1_hoaks","fp","fn"]])
if (D / "D4_UJI_SIGNIFIKANSI.csv").exists():
    display(pd.read_csv(D / "D4_UJI_SIGNIFIKANSI.csv")[["model","delta_observed","delta_ci_95","p_value_laporan"]])
if (D / "D4_UJI_AGEN_SISTEM.csv").exists():
    display(pd.read_csv(D / "D4_UJI_AGEN_SISTEM.csv")[["c_fn","sistem","coverage","galat_diputuskan","fp","fn","ditolak","kerugian_harapan","kerugian_harapan_ci95"]])
    display(pd.read_csv(D / "D4_UJI_AGEN_LAPISAN.csv"))
run("scripts/anggota_d/d2_status_uji.py", "--dataset", DATASET_PATH)

### 2d. Unduh semua hasil
Salin isi zip ke `results/anggota_d/` di laptop, lalu commit `uji_log.csv` bersama hasil UJI.

In [ ]:
zpath = Path("/content/HASIL_D4_UJI.zip") if Path("/content").exists() else Path("HASIL_D4_UJI.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(D.iterdir()):
        if p.name.startswith(("D4_", "uji_log", "experiments_uji", "PEMBEKUAN_UJI")):
            z.write(p, f"results/anggota_d/{p.name}")
print("Zip:", zpath)
try:
    from google.colab import files
    files.download(str(zpath))
except ImportError:
    pass